# Phase 1 — Budget-matched Random Search (SINGLE-FILE VERSION)

All pipeline code is embedded below, so no `src/` folder is needed.

**You still need three things next to this notebook:**

```
your_folder/
├── this_notebook.ipynb
├── data/     leukemia.mat, colon.mat
└── results/  final_raw_results.json
             breast_cancer_matched_result.json
             matched_seed_control.json
```

The `.mat` files are binary and the `.json` files are experiment output, so neither can
be embedded in a notebook. Everything else is here.

**Trade-off you should know about.** The multi-file version imports the pipeline from
`src/`, which guarantees the notebook and the original experiments run identical code.
Here the module code is *copied* verbatim into cells 2-6. It is byte-identical today, but
if you ever edit `src/`, this file will not follow. Treat the multi-file version as the
source of truth and this one as a convenience copy.

**Pre-registered decisions (locked before any result was seen):**
1. Budget is measured, not assumed (DEAP re-evaluates only invalidated individuals;
   measured seed 0: Breast Cancer 745, Colon 840, Leukemia 840).
2. Random Search returns a *front*: the non-dominated set of its samples, reduced by the
   same front-union and knee rules.
3. Cardinality `k ~ Uniform{1..min(p,50)}`, then k features uniformly without replacement.
4. Chance baselines recomputed per method from that method's own subset sizes.
5. All comparisons descriptive; no significance test.


## 1. Environment and all imports

In [ ]:
import sys, platform, json, time, random, warnings
from pathlib import Path
import numpy as np
import scipy.io as sio
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, LeaveOneGroupOut, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.exceptions import ConvergenceWarning
from deap import base, creator, tools
warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("Python:", sys.version.split()[0], "|", platform.system())
import numpy, scipy, sklearn, deap
for m in (numpy, scipy, sklearn, deap): print(f"{m.__name__:>12}: {m.__version__}")
# install with:  pip install numpy scipy scikit-learn deap

# SEEDING POLICY
#  bootstrap draws: np.random.default_rng(0) drawn 10x in sequence (reproduces the
#                   exact draws used by the saved NSGA-II runs)
#  NSGA-II        : seeds 0-9 / 100-109      Random Search: 1000-1009 / 2000-2009

## 2. CONFIG

In [ ]:
PROJ_ROOT = Path.cwd()
if not (PROJ_ROOT/"data").exists() and (PROJ_ROOT.parent/"data").exists():
    PROJ_ROOT = PROJ_ROOT.parent
DATA_DIR    = PROJ_ROOT/"data"
RESULTS_DIR = PROJ_ROOT/"results"; RESULTS_DIR.mkdir(exist_ok=True)
OUT_FILE    = RESULTS_DIR/"phase1_random_search.json"
BUDGET_FILE = RESULTS_DIR/"phase1_measured_budget.json"

DATASETS     = ["breast_cancer", "colon", "leukemia"]
N_RUNS       = 10
K_MAX_CAP    = 50
BUDGET_SEEDS = [0, 1, 2]
RS_SEED_BASE = {"bootstrap": 1000, "fixed": 2000}
POP_SIZE, N_GEN = 40, 20

assert DATA_DIR.exists(), f"data/ not found under {PROJ_ROOT}"
print("PROJ_ROOT:", PROJ_ROOT)

## 3. Pipeline — data loaders *(copied from `src/data.py`)*

In [ ]:
"""
Dataset loaders. Deliberately return RAW X, y with zero preprocessing.
All scaling/imputation must happen later, inside CV folds only (see nested_cv.py).
Loading raw here is itself part of leakage prevention: if preprocessing lived
here, it would be trivial to accidentally fit it on the full dataset once.
"""
from pathlib import Path

# DATA_DIR comes from the CONFIG cell above


def load_breast_cancer_wisconsin():
    """Debug dataset. 569 samples, 30 features, binary, well-balanced-ish (357/212)."""
    d = load_breast_cancer()
    X = d.data.astype(float)
    y = d.target.astype(int)
    return X, y, "breast_cancer_wisconsin"


def _load_skfeature_mat(name):
    path = DATA_DIR / f"{name}.mat"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Download from "
            f"https://raw.githubusercontent.com/jundongl/scikit-feature/master/skfeature/data/{name}.mat"
        )
    d = sio.loadmat(path)
    X = np.asarray(d["X"], dtype=float)
    y = np.asarray(d["Y"]).ravel()
    y = (y == y.max()).astype(int)  # map {-1,1} -> {0,1}; deterministic, not data-dependent stat
    return X, y


def load_leukemia():
    """ALLAML / Leukemia (Golub et al. 1999), via scikit-feature (ASU) repo.
    72 samples, 7070 genes, classes 47/25. p/n ~ 99 -> primary p>>n dataset."""
    X, y = _load_skfeature_mat("leukemia")
    return X, y, "leukemia_allaml"


def load_colon():
    """Colon (Alon et al. 1999), via scikit-feature (ASU) repo.
    62 samples, 2000 genes, classes 40/22. p/n ~ 32 -> optional robustness check."""
    X, y = _load_skfeature_mat("colon")
    return X, y, "colon"


REGISTRY = {
    "breast_cancer": load_breast_cancer_wisconsin,
    "leukemia": load_leukemia,
    "colon": load_colon,
}


def load(name):
    X, y, tag = REGISTRY[name]()
    assert X.shape[0] == y.shape[0], "X/y sample count mismatch"
    assert set(np.unique(y)) <= {0, 1}, "expected binary labels 0/1"
    return X, y, tag

## 4. Pipeline — LEAKAGE-CRITICAL cross-validation *(copied from `src/nested_cv.py`)*

Read this cell if you read only one. It enforces both leakage controls: the scaler is fit
on the inner-training fold only, and CV is group-aware so duplicate rows created by
bootstrap resampling never straddle train/validation.


In [ ]:
"""
LEAKAGE-CRITICAL MODULE. Read this file first when auditing the pipeline.

TWO distinct leakage risks are guarded here, not one:

(1) Scaling/preprocessing leakage: the scaler is always .fit() on the
    inner-training fold only, then .transform() on both train and
    validation folds. No statistic computed from validation data ever
    touches training.

(2) Duplicate-sample leakage from bootstrap resampling: when X,y come from
    a WITH-REPLACEMENT bootstrap draw (see bootstrap.py), the same original
    sample can appear multiple times as different rows. A naive CV split
    on row position can then put one copy of a patient in the training
    fold and another copy of THE SAME patient in the validation fold --
    the model has effectively already seen the "held-out" case. This is
    silent and does not raise any error; it just inflates accuracy and can
    distort which feature subsets look best. Fixed here via GroupKFold /
    LeaveOneGroupOut using the ORIGINAL (pre-resampling) sample index as
    the group label, so every duplicate of a given original sample is
    forced to the same side of every split. Empirically, about 40% of rows
    in a size-72 bootstrap are duplicates of an already-included sample
    (see audit note in conversation) -- this is not a rare edge case.

The feature mask is chosen by the GA using ONLY the fitness values this
module returns.
"""

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def make_cv(n_samples, n_splits=5, loo_threshold=100, seed=0, groups=None):
    """LOOCV for very small n, else stratified/grouped k-fold.
    groups: array of length n_samples giving the ORIGINAL sample identity
    for each row (needed when the data has been bootstrap-resampled with
    replacement, so duplicate rows of the same original sample never get
    split across train/validation -- see module docstring update below).
    If groups is None, every row is assumed to be its own unique sample
    (correct for un-resampled data)."""
    if groups is not None and len(np.unique(groups)) < n_samples:
        # duplicates present -> must use group-aware splitting
        n_unique = len(np.unique(groups))
        if n_unique <= loo_threshold:
            return LeaveOneGroupOut()
        return GroupKFold(n_splits=n_splits)
    if n_samples <= loo_threshold:
        return LeaveOneOut()
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)


def eval_mask(X, y, mask, cv, classifier_fn=None, groups=None):
    """Inner-CV balanced accuracy for one feature mask on one dataset.
    groups: passed straight to cv.split(); required (and must match what
    make_cv() was built with) whenever X,y come from a bootstrap resample
    with duplicate rows -- otherwise duplicate copies of the same original
    sample can straddle the train/validation boundary."""
    if mask.sum() == 0:
        return 0.0
    if classifier_fn is None:
        classifier_fn = lambda: KNeighborsClassifier(n_neighbors=5)

    Xs = X[:, mask]
    accs = []
    for tr_idx, va_idx in cv.split(Xs, y, groups=groups):
        Xtr, Xva = Xs[tr_idx], Xs[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)      # fit ONLY on training fold
        Xva = scaler.transform(Xva)          # validation fold only transformed

        if len(np.unique(ytr)) < 2:
            # degenerate fold (can happen with LOOCV): fall back to majority class
            pred = np.full(len(yva), ytr[0])
        else:
            clf = classifier_fn()
            clf.fit(Xtr, ytr)
            pred = clf.predict(Xva)

        accs.append(_balanced_acc(yva, pred))
    return float(np.mean(accs))


def _balanced_acc(y_true, y_pred):
    """Balanced accuracy, computed by hand (avoids sklearn's warning noise
    on single-sample LOOCV folds) -- macro-average of per-class recall over
    classes actually present in y_true for this fold."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    classes = np.unique(y_true)
    if len(classes) == 1:
        return float((y_pred == y_true).mean())
    recalls = []
    for c in classes:
        m = y_true == c
        recalls.append((y_pred[m] == c).mean())
    return float(np.mean(recalls))

## 5. Pipeline — NSGA-II *(copied from `src/ga.py`)*

In [ ]:
"""
NSGA-II over binary feature masks. Objectives: maximize inner-CV balanced
accuracy, minimize number of selected features. Kept separate from
nested_cv.py so the optimizer can be swapped/audited independently of the
leakage-critical fitness code.
"""

# DEAP's creator uses module-level globals; guard against re-registration
# if this module is imported more than once in one process (e.g. notebooks).
if not hasattr(creator, "FitnessMulti"):
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0))  # (maximize acc, minimize #features)
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMulti)


def build_toolbox(n_features, eval_fn, seed=0):
    """eval_fn: mask(np.bool array) -> accuracy (float). Feature-count
    objective is computed here, not passed in, so eval_fn only ever needs
    to know about accuracy -- keeps nested_cv.py single-purpose."""
    rng = random.Random(seed)
    tb = base.Toolbox()

    def init_individual():
        # start sparse: each gene True with small prob, so initial masks
        # aren't ~50% of thousands of genes (which would be a slow, bad start
        # for p=7070 problems)
        p_on = min(0.05, 20.0 / n_features)
        return creator.Individual([1 if rng.random() < p_on else 0 for _ in range(n_features)])

    tb.register("individual", init_individual)
    tb.register("population", tools.initRepeat, list, tb.individual)

    def evaluate(ind):
        mask = np.array(ind, dtype=bool)
        n_sel = int(mask.sum())
        if n_sel == 0:
            return (0.0, 0)
        acc = eval_fn(mask)
        return (acc, n_sel)

    tb.register("evaluate", evaluate)
    tb.register("mate", tools.cxUniform, indpb=0.5)
    tb.register("mutate", tools.mutFlipBit, indpb=1.0 / n_features)
    tb.register("select", tools.selNSGA2)
    return tb


def run_nsga2(n_features, eval_fn, pop_size=60, n_gen=40, seed=0, verbose=False):
    """Returns the final Pareto front as a list of (mask: np.bool array, acc: float, n_sel: int)."""
    if pop_size % 4 != 0:
        raise ValueError(f"pop_size must be divisible by 4 for selTournamentDCD (got {pop_size})")
    tb = build_toolbox(n_features, eval_fn, seed=seed)
    random.seed(seed)
    pop = tb.population(n=pop_size)
    for ind in pop:
        ind.fitness.values = tb.evaluate(ind)

    pop = tb.select(pop, len(pop))
    for gen in range(n_gen):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [tb.clone(ind) for ind in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < 0.9:
                tb.mate(c1, c2)
            tb.mutate(c1)
            tb.mutate(c2)
            del c1.fitness.values, c2.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = tb.evaluate(ind)
        pop = tb.select(pop + offspring, pop_size)
        if verbose and gen % 10 == 0:
            best_acc = max(ind.fitness.values[0] for ind in pop)
            print(f"  gen {gen:>3} best_acc={best_acc:.3f}")

    front = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    out = []
    seen = set()
    for ind in front:
        mask = tuple(ind)
        if mask in seen:
            continue
        seen.add(mask)
        acc, n_sel = ind.fitness.values
        out.append((np.array(ind, dtype=bool), acc, int(n_sel)))
    out.sort(key=lambda t: t[2])
    return out

## 6. Pipeline — stability metrics *(copied from `src/stability.py` and `src/stability_ci.py`)*

In [ ]:
"""
Nogueira, Sechidis & Brown (JMLR 2018) stability estimator, plus two
explicit ways to turn "one Pareto front per bootstrap" into "one binary
mask per bootstrap" (which is what the Nogueira estimator assumes as
input). This ambiguity was flagged before writing any code -- both
definitions are computed and logged; which one goes in the paper is a
decision for after we see real numbers, not before.

Nogueira formula (closed-form, O(M*p), avoids pairwise Jaccard):
  Given Z: (M runs) x (p features) binary matrix.
  k_i      = features selected in run i (row sum)
  kbar     = mean(k_i)
  phat_f   = mean over runs of Z[:,f]           (selection frequency of feature f)
  numer    = (1/p) * sum_f phat_f*(1-phat_f) * M/(M-1)      [unbiased per-feature variance]
  denom    = (kbar/p) * (1 - kbar/p)                         [variance under a null model]
  stability = 1 - numer/denom                                [ -> 1 = perfectly stable ]
Undefined (denom=0) when kbar=0 or kbar=p for every run; returns np.nan in that
degenerate case rather than dividing by zero.
"""


def nogueira_stability(Z):
    """Z: array-like, shape (M, p), binary. Returns (stability, detail dict)."""
    Z = np.asarray(Z, dtype=float)
    M, p = Z.shape
    if M < 2:
        raise ValueError("Nogueira stability needs at least 2 runs (bootstraps).")
    k = Z.sum(axis=1)
    kbar = k.mean()
    phat = Z.mean(axis=0)
    numer = (phat * (1 - phat)).mean() * (M / (M - 1))
    denom = (kbar / p) * (1 - kbar / p)
    if denom <= 1e-12:
        return float("nan"), dict(M=M, p=p, kbar=kbar, numer=numer, denom=denom, note="degenerate: kbar is 0 or p for all runs")
    stab = 1.0 - numer / denom
    return float(stab), dict(M=M, p=p, kbar=float(kbar), numer=float(numer), denom=float(denom))


def masks_to_matrix(masks, n_features):
    """List of boolean arrays (possibly ragged in the sense of coming from
    different individuals but all length n_features) -> (M, p) matrix."""
    Z = np.zeros((len(masks), n_features), dtype=int)
    for i, m in enumerate(masks):
        Z[i, :] = np.asarray(m, dtype=int)
    return Z


def front_level_mask(front, n_features):
    """DEFINITION: a feature counts as 'selected by this bootstrap' if it
    appears in AT LEAST ONE Pareto-optimal subset from that bootstrap's run.
    front: list of (mask, acc, n_sel) tuples as returned by ga.run_nsga2.
    This is a union, not a weighted vote -- documented explicitly because
    it is a real modeling choice, not the only reasonable one."""
    if len(front) == 0:
        return np.zeros(n_features, dtype=int)
    union = np.zeros(n_features, dtype=bool)
    for mask, acc, n_sel in front:
        union |= mask
    return union.astype(int)


def knee_point_mask(front):
    """DEFINITION: point-level representative = knee of the Pareto front,
    found by normalizing both objectives to [0,1] over THIS front and
    minimizing distance to the ideal point (acc=1, n_sel=0).
    front: list of (mask, acc, n_sel)."""
    if len(front) == 0:
        return None
    if len(front) == 1:
        return front[0][0]
    accs = np.array([a for _, a, _ in front])
    nsels = np.array([n for _, _, n in front])
    a_rng = accs.max() - accs.min()
    n_rng = nsels.max() - nsels.min()
    acc_norm = (accs - accs.min()) / a_rng if a_rng > 0 else np.zeros_like(accs)
    nsel_norm = (nsels - nsels.min()) / n_rng if n_rng > 0 else np.zeros_like(nsels)
    dist = np.sqrt((1 - acc_norm) ** 2 + nsel_norm ** 2)
    return front[int(np.argmin(dist))][0]


def compute_both_stabilities(fronts, n_features):
    """fronts: list (length = #bootstraps) of Pareto fronts (each a list of
    (mask, acc, n_sel)). Returns dict with both operationalizations."""
    front_masks = [front_level_mask(f, n_features) for f in fronts]
    point_masks = [knee_point_mask(f) for f in fronts]
    point_masks = [m for m in point_masks if m is not None]

    Z_front = masks_to_matrix(front_masks, n_features)
    Z_point = masks_to_matrix(point_masks, n_features)

    stab_front, detail_front = nogueira_stability(Z_front)
    stab_point, detail_point = nogueira_stability(Z_point)
    return {
        "front_level": {"stability": stab_front, **detail_front},
        "point_level": {"stability": stab_point, **detail_point},
    }


# ---- from stability_ci.py ----
"""
Confidence interval for the Nogueira et al. (JMLR 2018) stability estimator.

Two independent methods, deliberately both implemented so they can be
cross-checked against each other (if an asymptotic CI and a bootstrap CI
disagree badly at M=10, that itself is the finding -- it means M is too
small for the asymptotic approximation and we must report the bootstrap one).

METHOD A -- asymptotic (Nogueira et al., Sec. 5).
  Treats the M runs as iid draws. The estimator is
      stab = 1 - (mean_f s_f^2) / (kbar/p * (1 - kbar/p))
  where s_f^2 = M/(M-1) * phat_f(1-phat_f) is the unbiased per-feature
  variance. The variance of stab is obtained by the delta method over the
  per-run contributions; we compute it empirically via the influence-function
  form, which is what makes it valid without assuming a distribution on Z.

METHOD B -- nonparametric bootstrap over RUNS.
  Resample the M runs (rows of Z) with replacement B times, recompute
  stability each time, take empirical percentiles. Makes no asymptotic
  assumption. With M=10 this is coarse (only 10 distinct rows to draw from)
  but it is honest about that coarseness rather than hiding it.

IMPORTANT LIMITATION, applies to both: with M=10 runs, ANY interval on a
stability coefficient will be wide. These CIs exist to be reported honestly,
not to manufacture significance.
"""


def _nogueira_stability_dup(Z):  # duplicate name; the version above is used
    """Point estimate. Z: (M, p) binary. Returns (stability, detail)."""
    Z = np.asarray(Z, dtype=float)
    M, p = Z.shape
    if M < 2:
        raise ValueError("need >= 2 runs")
    k = Z.sum(axis=1)
    kbar = k.mean()
    phat = Z.mean(axis=0)
    numer = (phat * (1 - phat)).mean() * (M / (M - 1))
    denom = (kbar / p) * (1 - kbar / p)
    if denom <= 1e-12:
        return float("nan"), dict(M=M, p=p, kbar=kbar, note="degenerate")
    return float(1.0 - numer / denom), dict(M=M, p=p, kbar=float(kbar))


def _stability_from_rows(Z):
    s, _ = nogueira_stability(Z)
    return s


def ci_asymptotic(Z, alpha=0.05):
    """METHOD A. Influence-function / jackknife-style asymptotic interval.
    We use the jackknife over runs to estimate Var(stab), which is a
    consistent estimator of the asymptotic variance and avoids having to
    hand-differentiate the ratio."""
    Z = np.asarray(Z, dtype=float)
    M = Z.shape[0]
    full = _stability_from_rows(Z)
    if not np.isfinite(full):
        return (float("nan"), float("nan"), float("nan"))

    # leave-one-run-out replicates
    reps = []
    for i in range(M):
        Zi = np.delete(Z, i, axis=0)
        s = _stability_from_rows(Zi)
        if np.isfinite(s):
            reps.append(s)
    reps = np.array(reps)
    if len(reps) < 2:
        return (full, float("nan"), float("nan"))

    m = len(reps)
    var_jack = (m - 1) / m * np.sum((reps - reps.mean()) ** 2)
    se = float(np.sqrt(max(var_jack, 0.0)))
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    return (float(full), float(full - z * se), float(full + z * se))


def ci_bootstrap(Z, alpha=0.05, B=5000, seed=0):
    """METHOD B. Nonparametric bootstrap over runs (rows)."""
    Z = np.asarray(Z, dtype=float)
    M = Z.shape[0]
    rng = np.random.default_rng(seed)
    full = _stability_from_rows(Z)
    vals = []
    for _ in range(B):
        idx = rng.integers(0, M, M)
        Zb = Z[idx]
        s = _stability_from_rows(Zb)
        if np.isfinite(s):
            vals.append(s)
    if len(vals) < 100:
        return (float(full), float("nan"), float("nan"))
    vals = np.array(vals)
    lo = float(np.percentile(vals, 100 * alpha / 2))
    hi = float(np.percentile(vals, 100 * (1 - alpha / 2)))
    return (float(full), lo, hi)


def ci_both(Z, alpha=0.05, seed=0):
    est, a_lo, a_hi = ci_asymptotic(Z, alpha)
    _, b_lo, b_hi = ci_bootstrap(Z, alpha, seed=seed)
    return {
        "stability": est,
        "ci_asymptotic": [a_lo, a_hi],
        "ci_bootstrap": [b_lo, b_hi],
        "alpha": alpha,
    }

## 7. Pipeline — bootstrap helper *(copied from `src/bootstrap.py`)*

In [ ]:
def stratified_bootstrap_indices(y, rng):
    """With-replacement resample drawn separately within each class, so the class
    ratio is preserved and a small dataset cannot lose its minority class."""
    idx = np.arange(len(y)); out = []
    for c in np.unique(y):
        c_idx = idx[y == c]
        out.append(rng.choice(c_idx, size=len(c_idx), replace=True))
    return np.concatenate(out)

## 8. Measure the NSGA-II budget

Instruments the real `eval_fn`. Reports the spread across seeds; the matched Random Search
budget is the mean rounded up, per dataset.


In [ ]:
def measure_budget(ds, seeds):
    X, y, tag = load(ds); p = X.shape[1]
    counts, exposure = [], []
    for s in seeds:
        rng = np.random.default_rng(0)
        bidx = stratified_bootstrap_indices(y, rng)      # first draw
        Xb, yb = X[bidx], y[bidx]
        cv = make_cv(len(yb), seed=0, groups=bidx)
        st = {"n": 0, "seen": np.zeros(p, bool)}
        def ef(mask, Xb=Xb, yb=yb, cv=cv, g=bidx, st=st):
            st["n"] += 1; st["seen"] |= mask
            return eval_mask(Xb, yb, mask, cv, groups=g)
        run_nsga2(p, ef, pop_size=POP_SIZE, n_gen=N_GEN, seed=s)
        counts.append(st["n"]); exposure.append(int(st["seen"].sum()))
    return counts, exposure

if BUDGET_FILE.exists():
    budget = json.load(open(BUDGET_FILE))
    print("loaded cached budget measurement")
else:
    budget = {}
    for ds in DATASETS:
        t0 = time.time()
        counts, exposure = measure_budget(ds, BUDGET_SEEDS)
        X, _, _ = load(ds); p = X.shape[1]
        budget[ds] = {"counts": counts, "mean": float(np.mean(counts)),
                      "budget": int(np.ceil(np.mean(counts))),
                      "min": int(min(counts)), "max": int(max(counts)),
                      "p": p, "exposure": exposure,
                      "exposure_pct": [round(100*e/p, 1) for e in exposure]}
        print(f"{ds:>14}: evals {counts} -> budget {budget[ds]['budget']} | "
              f"exposure {budget[ds]['exposure_pct']}% [{time.time()-t0:.0f}s]", flush=True)
    json.dump(budget, open(BUDGET_FILE, "w"), indent=2)

for ds in DATASETS:
    print(f"  {ds:>14}: matched RS budget = {budget[ds]['budget']} evaluations")

## 9. Random Search

Same data, preprocessing, CV, fitness function, run count and OOB evaluation as NSGA-II.
The **only** difference is the search mechanism.


In [ ]:
def sample_mask(p, rng, k_max_cap=K_MAX_CAP):
    """Decision 3: k ~ Uniform{1..min(p, 50)}, then k features uniformly w/o replacement."""
    k_max = min(p, k_max_cap)
    k = int(rng.integers(1, k_max + 1))
    mask = np.zeros(p, dtype=bool)
    mask[rng.choice(p, k, replace=False)] = True
    return mask

def pareto_front(samples):
    """Non-dominated set w.r.t. (maximize acc, minimize n_sel).
    samples: list of (mask, acc, n_sel). Returns the same tuple format as
    ga.run_nsga2 so stability.py's reductions apply unchanged."""
    front = []
    for i, (mi, ai, ki) in enumerate(samples):
        dominated = False
        for j, (mj, aj, kj) in enumerate(samples):
            if i == j:
                continue
            if (aj >= ai and kj <= ki) and (aj > ai or kj < ki):
                dominated = True; break
        if not dominated:
            front.append((mi, ai, ki))
    # de-duplicate identical objective pairs, keep sorted by size like run_nsga2 does
    seen, out = set(), []
    for m, a, k in sorted(front, key=lambda t: t[2]):
        key = (a, k)
        if key in seen:
            continue
        seen.add(key); out.append((m, a, k))
    return out

def run_random_search(X, y, bidx, budget, seed, p):
    """One RS run on one (already drawn) bootstrap sample."""
    rng = np.random.default_rng(seed)
    Xb, yb = X[bidx], y[bidx]
    cv = make_cv(len(yb), seed=0, groups=bidx)
    samples = []
    for _ in range(budget):
        m = sample_mask(p, rng)
        acc = eval_mask(Xb, yb, m, cv, groups=bidx)
        samples.append((m, acc, int(m.sum())))
    return pareto_front(samples), len(samples)

## 10. Out-of-bag evaluation

Mirrors `results/oob_full.py`. Cell 8 asserts it reproduces the published NSGA-II OOB
values, which validates this path before it is used on Random Search.


In [ ]:
def balanced_acc(yt, yp):
    cs = np.unique(yt)
    if len(cs) < 2:
        return float((yp == yt).mean())
    return float(np.mean([(yp[yt == c] == c).mean() for c in cs]))

def oob_accuracy(X, y, bidx, features):
    """Train on in-bag rows restricted to `features`, evaluate on out-of-bag samples."""
    oob = np.setdiff1d(np.arange(len(y)), np.unique(bidx))
    if len(oob) < 5 or len(np.unique(y[oob])) < 2:
        return None, len(oob)
    Xtr, ytr = X[np.ix_(bidx, features)], y[bidx]
    Xte, yte = X[np.ix_(oob, features)], y[oob]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    clf = KNeighborsClassifier(n_neighbors=min(5, len(ytr))); clf.fit(Xtr, ytr)
    return balanced_acc(yte, clf.predict(Xte)), len(oob)

## 11. Run — checkpointed after each (dataset, condition)

In [ ]:
results = json.load(open(OUT_FILE)) if OUT_FILE.exists() else {}

for ds in DATASETS:
    X, y, tag = load(ds); p = X.shape[1]
    B = budget[ds]["budget"]

    for cond in ["bootstrap", "fixed"]:
        key = f"{ds}|{cond}"
        if key in results:
            print(f"SKIP {key} (done)"); continue

        # Reproduce EXACTLY the bootstrap draws used by the saved NSGA-II runs.
        rng_boot = np.random.default_rng(0)
        draws = [stratified_bootstrap_indices(y, rng_boot) for _ in range(N_RUNS)]
        if cond == "fixed":
            draws = [draws[0]] * N_RUNS          # decision: same single draw every run

        runs, t0 = [], time.time()
        for i in range(N_RUNS):
            seed = RS_SEED_BASE[cond] + i
            bidx = draws[i]
            front, n_evals = run_random_search(X, y, bidx, B, seed, p)
            knee = knee_point_mask(front)
            feats = np.where(knee)[0].tolist() if knee is not None else []
            oob_acc, oob_n = oob_accuracy(X, y, bidx, feats) if feats else (None, 0)
            runs.append({
                "dataset": ds, "method": "random_search", "condition": cond,
                "seed": int(seed), "bootstrap_id": i,
                "n_evals": int(n_evals), "front_size": len(front),
                "knee_features": feats, "knee_n_features": len(feats),
                "knee_fitness": float([a for m, a, k in front
                                       if np.array_equal(m, knee)][0]) if feats else None,
                "oob_accuracy": oob_acc, "oob_set_size": int(oob_n),
                "front": [{"n_sel": int(k), "acc": float(a),
                           "features": np.where(m)[0].tolist()} for m, a, k in front],
            })
            print(f"  [{ds}/{cond}] run {i+1}/{N_RUNS} front={len(front)} "
                  f"k={len(feats)} oob={oob_acc if oob_acc is None else round(oob_acc,3)} "
                  f"[{time.time()-t0:.0f}s]", flush=True)

        results[key] = {"budget": B, "n_runs": N_RUNS, "p": p, "runs": runs}
        json.dump(results, open(OUT_FILE, "w"), indent=2)
        print(f"--- CHECKPOINT {key} saved ---", flush=True)

print("ALL DONE ->", OUT_FILE)

## 12. Sanity checks — assertions that stop the notebook if violated

In [ ]:
import itertools

# A. OOB path reproduces the PUBLISHED NSGA-II values (validates cell 6 before we trust it)
published = {"breast_cancer": 0.930, "colon": 0.680, "leukemia": 0.774}
nsga_raw = json.load(open(RESULTS_DIR/"final_raw_results.json"))
bc_raw   = json.load(open(RESULTS_DIR / "breast_cancer_matched_result.json"))
for ds in DATASETS:
    X, y, _ = load(ds)
    runs = (bc_raw if ds == "breast_cancer" else nsga_raw[ds])["bootstrap_fixed"]["raw_runs"]
    rng_b = np.random.default_rng(0); accs = []
    for r in runs:
        bidx = stratified_bootstrap_indices(y, rng_b)
        front = [s for s in (r["front"] if isinstance(r, dict) else r) if s["n_sel"] > 0]
        if not front: continue
        tup = [(np.isin(np.arange(X.shape[1]), s["features"]), s["acc"], s["n_sel"]) for s in front]
        knee = knee_point_mask(tup)
        a, _ = oob_accuracy(X, y, bidx, np.where(knee)[0].tolist())
        if a is not None: accs.append(a)
    got = float(np.median(accs))
    assert abs(got - published[ds]) < 0.002, f"OOB path mismatch {ds}: {got} vs {published[ds]}"
    print(f"  OK  OOB path reproduces {ds}: {got:.3f}")

# B. Budget actually matched
for ds in DATASETS:
    for cond in ["bootstrap", "fixed"]:
        for r in results[f"{ds}|{cond}"]["runs"]:
            assert r["n_evals"] == budget[ds]["budget"], f"budget mismatch {ds}/{cond}"
print("  OK  every RS run used the measured NSGA-II budget")

# C. Cardinality distribution is the pre-registered one
for ds in DATASETS:
    p = budget[ds]["p"]; kmax = min(p, K_MAX_CAP)
    ks = [s["n_sel"] for c in ["bootstrap","fixed"] for r in results[f"{ds}|{c}"]["runs"] for s in r["front"]]
    assert min(ks) >= 1 and max(ks) <= kmax, f"cardinality out of range {ds}: {min(ks)}-{max(ks)}"
print("  OK  all sampled cardinalities within 1..min(p,50)")

# D. No OOB sample leaked into its own training bootstrap
for ds in DATASETS:
    X, y, _ = load(ds)
    rng_b = np.random.default_rng(0)
    for i in range(N_RUNS):
        bidx = stratified_bootstrap_indices(y, rng_b)
        oob = np.setdiff1d(np.arange(len(y)), np.unique(bidx))
        assert len(np.intersect1d(oob, np.unique(bidx))) == 0
print("  OK  out-of-bag sets are disjoint from their training bootstraps")

# E. Feature indices valid
for ds in DATASETS:
    p = budget[ds]["p"]
    for cond in ["bootstrap","fixed"]:
        for r in results[f"{ds}|{cond}"]["runs"]:
            for s in r["front"]:
                assert all(0 <= f < p for f in s["features"])
                assert len(set(s["features"])) == s["n_sel"]
print("  OK  feature indices valid and duplicate-free")

# F. Reproducibility: same seed -> same output
ds = "breast_cancer"; X, y, _ = load(ds); p = X.shape[1]
rng_b = np.random.default_rng(0); bidx0 = stratified_bootstrap_indices(y, rng_b)
f1, _ = run_random_search(X, y, bidx0, 50, 12345, p)
f2, _ = run_random_search(X, y, bidx0, 50, 12345, p)
assert [(a, k) for m, a, k in f1] == [(a, k) for m, a, k in f2]
print("  OK  reruns with the same seed are identical")

print("\nALL SANITY CHECKS PASSED")

## 13. What to send back

Send **`results/phase1_random_search.json`** and **`results/phase1_measured_budget.json`**,
plus the console output of cells 4, 7 and 8.

Do not edit any pre-registered decision to make a result look better. If a sanity check
fails, send the error rather than working around it.
